In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, roc_auc_score

In [2]:
df = pd.read_csv("../outputs/final_dataset.csv")
df = df.dropna()
print(df["기준_년분기_코드"].value_counts().sort_index())

기준_년분기_코드
20201    8429
20202    8473
20203    8492
20204    8491
20211    8705
20212    8707
20213    8715
20214    8714
20221    8698
20222    8700
20223    8696
20224    8691
20231    8678
20232    8668
20233    8669
20234    8653
20241    8647
20242    8641
20243    8619
20244    8603
20251    8563
20252    8578
20253    8585
Name: count, dtype: int64


In [3]:
df = df.sort_values(["행정동코드", "통합카테고리", "기준_년분기_코드"])

# ── 트렌드 피처 ──────────────────────────────────────────────────
df["매출_증감률"] = (
    df.groupby(["행정동코드", "통합카테고리"])["당월매출합"]
    .pct_change()
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)
df["매출_2분기평균"] = df.groupby(["행정동코드", "통합카테고리"])["당월매출합"].transform(
    lambda x: x.rolling(2, min_periods=1).mean()
)
df["매출_4분기평균"] = df.groupby(["행정동코드", "통합카테고리"])["당월매출합"].transform(
    lambda x: x.rolling(4, min_periods=1).mean()
)
df["매출_4분기std"] = (
    df.groupby(["행정동코드", "통합카테고리"])["당월매출합"]
    .transform(lambda x: x.rolling(4, min_periods=2).std())
    .fillna(0)
)
df["매출_모멘텀"] = df["매출_2분기평균"] - df["매출_4분기평균"]

df["유동_증감률"] = (
    df.groupby(["행정동코드", "통합카테고리"])["총유동인구"]
    .pct_change()
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)
df["유동_4분기평균"] = df.groupby(["행정동코드", "통합카테고리"])["총유동인구"].transform(
    lambda x: x.rolling(4, min_periods=1).mean()
)

# 객단가 트렌드 (가격대 변화 추이)
df["객단가_증감률"] = (
    df.groupby(["행정동코드", "통합카테고리"])["객단가"]
    .pct_change()
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)
df["객단가_4분기평균"] = df.groupby(["행정동코드", "통합카테고리"])["객단가"].transform(
    lambda x: x.rolling(4, min_periods=1).mean()
)

# 직전 3분기 중 성장한 횟수
df["연속_성장횟수"] = (
    df.groupby(["행정동코드", "통합카테고리"])["매출_증감률"]
    .transform(lambda x: (x > 0).rolling(3, min_periods=1).sum())
    .fillna(0)
)

# ── 다음 분기 타깃 ────────────────────────────────────────────────
df["다음분기_매출"] = df.groupby(["행정동코드", "통합카테고리"])["당월매출합"].shift(-1)
df["다음분기_증감률"] = (
    (df["다음분기_매출"] - df["당월매출합"]) / df["당월매출합"].replace(0, pd.NA)
)

# ── 라벨: 업종×분기 내 상대 성장 ─────────────────────────────────
df["label"] = (
    df.groupby(["통합카테고리", "기준_년분기_코드"])["다음분기_증감률"]
    .transform(lambda x: (x >= x.median()).astype(int))
)

labeled = df[df["다음분기_매출"].notna() & df["다음분기_증감률"].notna()].copy()

print("label 분포:")
print(labeled["label"].value_counts())
print(f"\n학습 가능 샘플 수: {len(labeled):,}")

label 분포:
label
1    94818
0    94557
Name: count, dtype: int64

학습 가능 샘플 수: 189,375


In [4]:
le = LabelEncoder()
labeled["업종_encoded"] = le.fit_transform(labeled["통합카테고리"])
labeled["분기번호"] = labeled["기준_년분기_코드"] % 10  # 계절성 반영 (1~4)

feature_cols = [
    # ── 유동인구 ──────────────────────────────────────────────────
    "총유동인구", "유동_4분기평균", "유동_증감률",
    "유동_20대비율", "유동_30대비율", "유동_40대비율", "유동_50대비율",
    "유동_여성비율",

    # ── 매출 ──────────────────────────────────────────────────────
    "당월매출합",
    "매출_증감률", "매출_2분기평균", "매출_4분기평균",
    "매출_4분기std", "매출_모멘텀",

    # ── 가격대 지표 ───────────────────────────────────────────────
    "객단가", "객단가_4분기평균", "객단가_증감률",

    # ── 연령대별 매출 비율 ─────────────────────────────────────────
    "매출_20대비율", "매출_30대비율", "매출_40대비율",
    "매출_50대비율", "매출_60대이상비율",

    # ── 성별 매출 비율 ────────────────────────────────────────────
    "매출_남성비율", "매출_여성비율",

    # ── 시간/요일 패턴 ────────────────────────────────────────────
    "매출_주말비율", "매출_점심비율", "매출_저녁비율", "매출_심야비율",

    # ── 업종 경쟁/포화 ────────────────────────────────────────────
    "업종_점포당매출", "업종_매출점유율",
    "경쟁강도", "업종_포화도",

    # ── 복합 지표 ─────────────────────────────────────────────────
    "MZ_차이", "유동대비매출", "점포대비유동",

    # ── 직장·주거인구 ─────────────────────────────────────────────
    "총_직장_인구_수",
    "직장_20대_비율", "직장_30대_비율", "직장_40대_비율",
    "직장_여성비율",
    "주거인구",

    # ── 개업/폐업/프랜차이즈 ──────────────────────────────────────
    "개업_율_평균", "폐업_률_평균", "프랜차이즈_점포수",

    # ── 메타 ──────────────────────────────────────────────────────
    "업종_encoded", "분기번호",

    # ── 성장 모멘텀 ───────────────────────────────────────────────
    "연속_성장횟수",
]

X = labeled[feature_cols]
y = labeled["label"]

all_quarters = sorted(df["기준_년분기_코드"].unique())
quarters = sorted(labeled["기준_년분기_코드"].unique())

train_mask = labeled["기준_년분기_코드"] < quarters[-1]
test_mask  = labeled["기준_년분기_코드"] == quarters[-1]

X_train, y_train = X[train_mask], y[train_mask]
X_test,  y_test  = X[test_mask],  y[test_mask]

print(f"훈련: {len(X_train):,}개 ({quarters[0]} ~ {quarters[-2]})")
print(f"테스트: {len(X_test):,}개 ({quarters[-1]} → {all_quarters[-1]})")
print(f"피처 수: {len(feature_cols)}")

훈련: 180,857개 (20201 ~ 20251)
테스트: 8,518개 (20252 → 20253)
피처 수: 47


In [5]:
model = lgb.LGBMClassifier(
    n_estimators=500,
    class_weight="balanced",
    random_state=42,
    verbose=-1
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, target_names=["감소", "성장"]))
print("AUC-ROC:", round(roc_auc_score(y_test, y_prob), 4))

# Feature 중요도
fi = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("\n[Feature 중요도]")
print(fi.round(4))

              precision    recall  f1-score   support

          감소       0.62      0.52      0.57      4248
          성장       0.59      0.68      0.63      4270

    accuracy                           0.60      8518
   macro avg       0.60      0.60      0.60      8518
weighted avg       0.60      0.60      0.60      8518

AUC-ROC: 0.6431

[Feature 중요도]
매출_증감률        695
객단가_증감률       595
매출_모멘텀        592
유동_증감률        579
객단가           445
매출_저녁비율       424
매출_40대비율      403
매출_주말비율       397
매출_남성비율       388
매출_점심비율       375
매출_30대비율      373
업종_점포당매출      346
주거인구          340
객단가_4분기평균     339
분기번호          338
매출_여성비율       335
매출_50대비율      329
유동_50대비율      325
유동_40대비율      321
유동_20대비율      314
매출_60대이상비율    314
매출_4분기std     313
매출_심야비율       313
유동_여성비율       312
업종_매출점유율      309
유동_30대비율      307
MZ_차이         306
직장_여성비율       297
직장_40대_비율     294
매출_20대비율      286
업종_encoded    285
개업_율_평균       283
업종_포화도        282
점포대비유동        274
총_직장_인구_수     264
직장_30대_비율   

In [6]:
from pathlib import Path

# ── 전체 최신 분기 창업 적합도 점수 계산 ────────────────────────────
latest_q = df["기준_년분기_코드"].max()
all_latest = df[df["기준_년분기_코드"] == latest_q].copy()

all_latest["업종_encoded"] = le.transform(all_latest["통합카테고리"])
all_latest["분기번호"] = all_latest["기준_년분기_코드"] % 10

# 모델 추론
all_latest["성장확률"] = model.predict_proba(all_latest[feature_cols])[:, 1]

# 업종별 순위 및 백분위
all_latest["업종내_순위"] = (
    all_latest.groupby("통합카테고리")["성장확률"]
    .rank(ascending=False, method="min")
    .astype(int)
)
all_latest["업종내_전체동수"] = (
    all_latest.groupby("통합카테고리")["성장확률"]
    .transform("count")
    .astype(int)
)
all_latest["상위_퍼센트"] = (
    all_latest["업종내_순위"] / all_latest["업종내_전체동수"] * 100
).round(1)

def to_grade(pct):
    if pct <= 25:   return "A"
    elif pct <= 50: return "B"
    elif pct <= 75: return "C"
    else:           return "D"

all_latest["등급"] = all_latest["상위_퍼센트"].apply(to_grade)
all_latest["성장확률_pct"] = (all_latest["성장확률"] * 100).round(1)

scores_df = all_latest[[
    "행정동명", "통합카테고리", "기준_년분기_코드",
    "성장확률_pct", "업종내_순위", "업종내_전체동수", "상위_퍼센트", "등급"
]].copy()

SCORES_PATH = Path("../outputs/scores.csv")
scores_df.to_csv(SCORES_PATH, index=False, encoding="utf-8-sig")

print(f"저장 완료: {SCORES_PATH}")
print(f"총 {len(scores_df):,}개 (행정동×업종)")
print(scores_df[scores_df["행정동명"] == "역삼1동"].sort_values("업종내_순위").to_string())


저장 완료: ../outputs/scores.csv
총 8,585개 (행정동×업종)
        행정동명    통합카테고리  기준_년분기_코드  성장확률_pct  업종내_순위  업종내_전체동수  상위_퍼센트 등급
235543  역삼1동        주점      20253      68.1      29       394     7.4  A
235534  역삼1동      애완동물      20253      57.6      43       145    29.7  B
235541  역삼1동        일식      20253      54.0      75       277    27.1  B
235525  역삼1동   B2B 서비스      20253      51.4     137       366    37.4  B
235535  역삼1동   양식/기타외식      20253      48.3     141       238    59.2  C
235531  역삼1동        숙박      20253      34.6     163       194    84.0  D
235544  역삼1동        중식      20253      49.4     186       356    52.2  C
235542  역삼1동     전자/통신      20253      39.2     186       304    61.2  C
235528  역삼1동    뷰티/화장품      20253      49.5     193       387    49.9  B
235529  역삼1동   생활용품 소매      20253      47.9     212       398    53.3  C
235537  역삼1동     오락/유흥      20253      40.5     263       359    73.3  C
235532  역삼1동    스포츠/레저      20253      40.2     267       387    69.0  C
2355

In [7]:
def recommend_dong_by_업종(업종명, df, model, le, feature_cols, top_n=10):
    """업종 클릭 → 창업하기 좋은 행정동 추천"""
    latest_q = df["기준_년분기_코드"].max()
    subset = df[(df["통합카테고리"] == 업종명) & (df["기준_년분기_코드"] == latest_q)].copy()

    if subset.empty:
        print(f"'{업종명}' 데이터가 없습니다.")
        return None

    subset["업종_encoded"] = le.transform(subset["통합카테고리"])
    subset["분기번호"] = subset["기준_년분기_코드"] % 10
    subset["성장확률"] = model.predict_proba(subset[feature_cols])[:, 1]

    result = (
        subset.nlargest(top_n, "성장확률")[["행정동명", "성장확률"]]
        .reset_index(drop=True)
    )
    result.index += 1

    print(f"\n[{업종명}] 창업 추천 행정동 TOP {top_n}\n")
    for i, row in result.iterrows():
        print(f"{i}위: {row['행정동명']} (성장확률: {row['성장확률']:.1%})")

    return result


def recommend_업종_by_dong(행정동명, df, model, le, feature_cols):
    """행정동 클릭 → 어울리는 업종 TOP 5 추천"""
    latest_q = df["기준_년분기_코드"].max()
    subset = df[(df["행정동명"] == 행정동명) & (df["기준_년분기_코드"] == latest_q)].copy()

    if subset.empty:
        print(f"'{행정동명}' 데이터가 없습니다.")
        return None

    subset["업종_encoded"] = le.transform(subset["통합카테고리"])
    subset["분기번호"] = subset["기준_년분기_코드"] % 10
    subset["성장확률"] = model.predict_proba(subset[feature_cols])[:, 1]

    result = (
        subset.nlargest(5, "성장확률")[["통합카테고리", "성장확률"]]
        .reset_index(drop=True)
    )
    result.index += 1

    print(f"\n[{행정동명}] 추천 업종 TOP 5\n")
    for i, row in result.iterrows():
        print(f"{i}위: {row['통합카테고리']} (성장확률: {row['성장확률']:.1%})")

    return result


# 테스트
print(df["통합카테고리"].unique())  # 카테고리명 확인
recommend_dong_by_업종("카페", df, model, le, feature_cols)
print()
recommend_업종_by_dong("역삼1동", df, model, le, feature_cols)

<StringArray>
[ 'B2B 서비스',      '미용실',    '분식/간식',   '뷰티/화장품',  '생활용품 소매',    '수리/세탁',
   '스포츠/레저',    '식품 소매',  '양식/기타외식',     '예술학원',    '의료/약국',    '의류/패션',
     '일반학원',       '일식',    '전자/통신',       '주점',       '중식',       '카페',
 '패스트푸드/치킨',      '편의점',       '한식',       '숙박',    '오락/유흥',     '애완동물']
Length: 24, dtype: str

[카페] 창업 추천 행정동 TOP 10

1위: 회기동 (성장확률: 97.0%)
2위: 사근동 (성장확률: 93.9%)
3위: 종로1.2.3.4가동 (성장확률: 91.4%)
4위: 안암동 (성장확률: 91.3%)
5위: 신촌동 (성장확률: 91.1%)
6위: 사직동 (성장확률: 89.6%)
7위: 가회동 (성장확률: 89.5%)
8위: 이문1동 (성장확률: 88.7%)
9위: 삼청동 (성장확률: 88.1%)
10위: 공릉2동 (성장확률: 86.2%)


[역삼1동] 추천 업종 TOP 5

1위: 주점 (성장확률: 68.1%)
2위: 애완동물 (성장확률: 57.6%)
3위: 일식 (성장확률: 54.0%)
4위: B2B 서비스 (성장확률: 51.4%)
5위: 뷰티/화장품 (성장확률: 49.5%)


,통합카테고리,성장확률
1,주점,0.680679
2,애완동물,0.576397
3,일식,0.539866
4,B2B 서비스,0.514301
5,뷰티/화장품,0.495026
